# Codice originale Tensorflow/keras

In [1]:
import os
import tensorflow as tf

print("=== TEST ARCHITETTURA DEFINITIVO (TENSORFLOW) ===")

# Verifiche prima di cominciare
cuda_var = os.environ.get("CUDA_VISIBLE_DEVICES")
print(f"1. Variabile CUDA_VISIBLE_DEVICES ereditata: {cuda_var}")

if cuda_var == "1":
    print("STATUS: SEI NEL SECONDO ESPERIMENTO (Scheda Fisica 1, Porta 8890)")
elif cuda_var == "0":
    print("STATUS: SEI NEL PRIMO ESPERIMENTO (Scheda Fisica 0, Porta 8888)")

print(f"\n2. TENSORFLOW:")
gpus = tf.config.list_physical_devices('GPU')
print(f"   - Quante GPU vede all'interno del suo recinto: {len(gpus)}")

if gpus:
    try:
        # Usa sempre gpus[0], perché per TF c'è solo una scheda in questo recinto
        details = tf.config.experimental.get_device_details(gpus[0])
        gpu_name = details.get('device_name', 'Nome non disponibile')
        print(f"   - Modello agganciato: {gpu_name}")
    except Exception as e:
        print(f"   - ID interno: {gpus[0].name}")
else:
    print("   - WARNING: Nessuna GPU rilevata (uso CPU).")

print(f"\nSystem Info - Kernel PID: {os.getpid()}")
print(f"Working folder: {os.getcwd()}")

I0000 00:00:1779032585.761137 1365679 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779032585.800091 1365679 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779032593.389877 1365679 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


=== 🕵️ TEST ARCHITETTURA DEFINITIVO (TENSORFLOW) ===
1. Variabile CUDA_VISIBLE_DEVICES ereditata: 1
STATUS: SEI NEL SECONDO ESPERIMENTO (Scheda Fisica 1, Porta 8890)

2. TENSORFLOW:
   - Quante GPU vede all'interno del suo recinto: 0
   - WARNING: Nessuna GPU rilevata (uso CPU).

System Info - Kernel PID: 1365679
Working folder: /home/emiliano/projects/project_1/Lab_XAI/Lab_XAI_tensorflow


E0000 00:00:1779032598.722412 1365679 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1779032598.747065 1365679 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [3]:
import tensorflow as tf

# Controlla se la lista delle GPU fisiche non è vuota
gpus = tf.config.list_physical_devices('GPU')

# Assegna la stringa corrispondente
device0 = "/GPU:0" if gpus else "/CPU:0"

print(f"Device: {device0}")

Device: /CPU:0


In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from keras.models import Sequential, save_model
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from joblib import dump

orbit_to_remove = []
with open('Data/orbit_to_remove') as file:
    for line in file:
        orbit_to_remove.append(float(line))
df = pd.read_csv("Data/MARSIS_historical_dataset.csv", sep=";")

frequency_to_keep = 4000000.0
df = df[df['FM_data_frequency'] == frequency_to_keep]
df = df[~df.FM_data_orbit_number.isin(orbit_to_remove)]
df['FM_data_solar_longitude_cos'] = np.cos(df['FM_data_solar_longitude'])
df['FM_data_solar_longitude_sin'] = np.sin(df['FM_data_solar_longitude'])

X = df.drop(columns=['FM_data_ephemeris_time', 'FM_data_F10_7_index', 'FM_data_frequency',
                     'FM_data_median_corrected_echo_power', 'FM_data_orbit_number', 'FM_data_peak_corrected_echo_power',
                     'FM_data_peak_distorted_echo_power', 'FM_data_peak_simulated_echo_power', 'FM_data_solar_longitude'])
col_names = X.columns.tolist()
X = X.to_numpy()

y = df['FM_data_peak_distorted_echo_power'].to_numpy()

kf = KFold(n_splits=10, shuffle=False)
kf.get_n_splits(X)

fold = -1
y_all_pred = np.zeros(y.shape)
f = open("MAE_nn_chrono.txt", "w")
t1 = time.time()
for train_index, test_index in kf.split(X):
    fold = fold + 1
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    f.write('TRAIN: ' + str(train_index)+'\n')
    f.write('TEST: ' + str(test_index)+'\n')
    dump(scaler, f'scaler_chrono_{fold}.save')

    # Neural Network
    es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)
    model = Sequential()
    model.add(Dense(800, input_dim=X.shape[1], activation='relu'))  # 300
    model.add(Dropout(0.5))
    model.add(Dense(400, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(200, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(100, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='relu'))
    model.compile(loss='mse', optimizer='adam', metrics=['mean_absolute_error'])
    model.fit(X_train, y_train, validation_split=0.1, epochs=1000, callbacks=[es])
    save_model(model, f'nn_chrono_model_{fold}.keras')
    # evaluate the model
    _, train_mse = model.evaluate(X_train, y_train, verbose=0)
    _, test_mse = model.evaluate(X_test, y_test, verbose=0)
    print('Train: %.3f, Test: %.3f' % (train_mse, test_mse))
    t2 = time.time()

    y_pred = model.predict(X_test).flatten()
    f.write(f'MAE = {mean_absolute_error(y_test,y_pred)}\n')
    f.write(f'MAPE = {mean_absolute_percentage_error(y_test, y_pred)}\n')
    f.write(f'MSE = {mean_squared_error(y_test, y_pred)}\n')
    f.write(f'Execution time = {t2 - t1}\n')
    for i in range(len(test_index)):
        y_all_pred[test_index[i]] = y_pred[i]

t3 = time.time()
f.write(f'\nGlobal MAE = {mean_absolute_error(y, y_all_pred)}')
f.write(f'\nGlobal MAPE = {mean_absolute_percentage_error(y, y_all_pred)}')
f.write(f'\nGlobal MSE = {mean_squared_error(y, y_all_pred)}')
f.write(f'\nTime = {t3-t1}')
f.close()
np.savetxt("nn_y_all_pred_chrono.txt", y_all_pred)

Epoch 1/1000


/home/emiliano/projects/project_1/Lab_XAI/.venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
W0000 00:00:1778942467.670057 1138427 cpu_allocator_impl.cc:82] Allocation of 111148560 exceeds 10% of free system memory.


48242/48242 ━━━━━━━━━━━━━━━━━━━━ 144s 3ms/step - loss: 13.8593 - mean_absolute_error: 2.8445 - val_loss: 11.4454 - val_mean_absolute_error: 2.5616
Epoch 2/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 144s 3ms/step - loss: 10.1462 - mean_absolute_error: 2.4668 - val_loss: 12.6610 - val_mean_absolute_error: 2.7762
Epoch 3/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 144s 3ms/step - loss: 9.6887 - mean_absolute_error: 2.4058 - val_loss: 11.7123 - val_mean_absolute_error: 2.6495
Epoch 4/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 145s 3ms/step - loss: 9.4517 - mean_absolute_error: 2.3724 - val_loss: 11.0984 - val_mean_absolute_error: 2.5662
Epoch 5/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 144s 3ms/step - loss: 9.3276 - mean_absolute_error: 2.3546 - val_loss: 12.4831 - val_mean_absolute_error: 2.6929
Epoch 6/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 143s 3ms/step - loss: 9.2216 - mean_absolute_error: 2.3403 - val_loss: 11.7554 - val_mean_absolute_error: 2.5970
Epoch 7/1000
48242/48242 ━━━━━━━━━━━━━━━━━━━━ 143s 3ms/st

W0000 00:00:1778945057.336832 1138427 cpu_allocator_impl.cc:82] Allocation of 123498432 exceeds 10% of free system memory.


Train: 2.053, Test: 2.445
5956/5956 ━━━━━━━━━━━━━━━━━━━━ 4s 701us/step


ValueError: setting an array element with a sequence.